In [1]:
import cv2
import numpy as np
import os
import pandas as pd

input_folder = "frames"

folders = [
    "1_Grayscale",
    "2_Denoised",
    "3_ShadowRemoved",
    "4_HistogramEqualized",
    "5_ContrastStretched",
    "6_Sharpened",
    "7_Final_Output"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

def estimate_noise_variance(gray):
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    noise = gray.astype(np.float32) - blur.astype(np.float32)
    return np.var(noise)

def detect_salt_pepper(gray, threshold=0.02):
    total_pixels = gray.size
    extreme_pixels = np.sum((gray == 0) | (gray == 255))
    return (extreme_pixels / total_pixels) > threshold

def remove_shadows(img, ksize=51):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    background = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
    return cv2.divide(img, background, scale=255)

def contrast_stretch(img):
    min_val = np.min(img)
    max_val = np.max(img)
    if max_val == min_val:
        return img.copy()
    stretched = (img - min_val) * (255.0 / (max_val - min_val))
    return stretched.astype(np.uint8)

def laplacian_variance(img):
    lap = cv2.Laplacian(img, cv2.CV_64F)
    return lap.var()


files = sorted(os.listdir(input_folder))
analysis_data = []

for file in files:
    if not file.lower().endswith((".jpg",".jpeg",".png",".bmp",".tif",".tiff")):
        continue

    path = os.path.join(input_folder, file)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    noise_var = estimate_noise_variance(gray)
    is_sp = detect_salt_pepper(gray)
    sharpness = laplacian_variance(gray)

    analysis_data.append([file, noise_var, is_sp, sharpness])

df = pd.DataFrame(analysis_data,
                  columns=["Frame","NoiseVariance","SaltPepper","Sharpness"])

mean_noise = df["NoiseVariance"].mean()
std_noise = df["NoiseVariance"].std()

low_noise = mean_noise - std_noise
high_noise = mean_noise + std_noise

def classify_noise(val):
    if val < low_noise:
        return "Low"
    elif val > high_noise:
        return "High"
    else:
        return "Moderate"

df["NoiseLevel"] = df["NoiseVariance"].apply(classify_noise)

mean_sharp = df["Sharpness"].mean()
std_sharp = df["Sharpness"].std()

blur_threshold = mean_sharp - 0.5 * std_sharp  

print("\nAdaptive Blur Threshold:", round(blur_threshold,2))

results = []

for index, row in df.iterrows():

    file = row["Frame"]
    path = os.path.join(input_folder, file)
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    cv2.imwrite(os.path.join("1_Grayscale", file), gray)

    if row["NoiseLevel"] != "Low":
        if row["SaltPepper"]:
            denoised = cv2.medianBlur(gray, 5)
            noise_type = "Salt & Pepper"
        else:
            denoised = cv2.GaussianBlur(gray, (5,5), 0)
            noise_type = "Gaussian"
    else:
        denoised = gray.copy()
        noise_type = "Low"

    cv2.imwrite(os.path.join("2_Denoised", file), denoised)

    shadow_free = remove_shadows(denoised)
    cv2.imwrite(os.path.join("3_ShadowRemoved", file), shadow_free)

    hist_eq = cv2.equalizeHist(shadow_free)
    cv2.imwrite(os.path.join("4_HistogramEqualized", file), hist_eq)

    stretched = contrast_stretch(hist_eq)
    cv2.imwrite(os.path.join("5_ContrastStretched", file), stretched)

    sharpness_value = laplacian_variance(stretched)

    if sharpness_value < blur_threshold:
        blur_status = "Blurred"
        lap = cv2.Laplacian(stretched, cv2.CV_64F)
        sharpened = stretched - lap
        sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    else:
        blur_status = "Sharp"
        sharpened = stretched.copy()

    cv2.imwrite(os.path.join("6_Sharpened", file), sharpened)

    cv2.imwrite(os.path.join("7_Final_Output", file), sharpened)

    contrast_before = np.std(gray)
    contrast_after = np.std(sharpened)

    results.append([file, noise_type, blur_status,
                    contrast_before, contrast_after])

print("\nAll images processed successfully.")

eval_df = pd.DataFrame(results,
        columns=["Frame","NoiseType","BlurStatus",
                 "ContrastBefore","ContrastAfter"])

print("\n FINAL PIPELINE RESULTS ")
print(eval_df)

print("\nBlur Distribution:")
print(eval_df["BlurStatus"].value_counts())

print("\nNoise Distribution:")
print(eval_df["NoiseType"].value_counts())


Adaptive Blur Threshold: 410.96

All images processed successfully.

 FINAL PIPELINE RESULTS 
                                Frame NoiseType BlurStatus  ContrastBefore  \
0       AlligatorCracking_frame_0.jpg  Gaussian      Sharp       36.268324   
1      AlligatorCracking_frame_20.jpg  Gaussian      Sharp       29.673586   
2      AlligatorCracking_frame_35.jpg  Gaussian      Sharp       28.744420   
3       AlligatorCracking_frame_4.jpg  Gaussian      Sharp       31.868419   
4      AlligatorCracking_frame_51.jpg  Gaussian      Sharp       32.110808   
5       AlligatorCracking_frame_9.jpg  Gaussian      Sharp       30.813715   
6          EdgeCracking3_frame_72.jpg  Gaussian      Sharp       24.160058   
7         EdgeCracking4_frame_123.jpg  Gaussian      Sharp       29.069681   
8         EdgeCracking4_frame_129.jpg  Gaussian      Sharp       29.767919   
9         EdgeCracking4_frame_130.jpg  Gaussian      Sharp       29.316910   
10        EdgeCracking4_frame_133.jpg  Gaussian